In [9]:
embeddings=embedding_manager.generate_embeddings(texts)

### Store in vector database
vectorstore.add_documents(chunks,embeddings)

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Batches: 100%|██████████| 63/63 [03:32<00:00,  3.37s/it]


Generated embeddings for 1999 texts. Embedding shape: (1999, 384)
Adding 1999 documents to the vector store...
Successfully added 1999 documents to the vector store.
Total documents in the collection after addition: 3998


Retriver pipeline form vectordb


In [52]:
class RAGRetreiver:
    """"RAG Retreiver class to retrieve relevant documents from a vector store based on a query."""
    def __init__(self,vector_store: VectorStore,embedding_manager:EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5,score_threshold: float=0.0) -> List[Dict[str,Any]]:  
          print(f"retrieving documents for query: {query}")
          print(f"Top K: {top_k}, Score Threshold: {score_threshold}")

          query_embedding=self.embedding_manager.generate_embeddings([query])[0]

          try:
               results=self.vector_store.collection.query(
                    query_embeddings=[query_embedding.tolist()],
                    n_results=top_k
               )

               retireved_docs=[]

               if results['documents'] and results['documents'][0]:
                    documents=results['documents'][0]
                    metadatas=results['metadatas'][0]
                    distances=results['distances'][0]
                    ids=results['ids'][0]

                    for i,(doc_id,document,metadata,distance) in enumerate(zip(ids,documents,metadatas,distances)):
                        similarity_score=1.0-distance

                        if similarity_score>=score_threshold:
                             retireved_docs.append({
                                  'id':doc_id,
                                  'content':document,
                                  'metadata':metadata,
                                  "similarity_score":similarity_score,
                                  'distance':distance,
                                  'rank':i+1
                             })
                    print(f"Retreived {len(retireved_docs)} documents for query: {query}")  
               else:
                    print(f"No documents found for query: {query}") 

               return retireved_docs
          except Exception as e:
               print(f"Error retrieving documents for query: {query}. Error: {e}")
               return [] 

rag_retriever=RAGRetreiver(vectorstore,embedding_manager)                         
                    

In [53]:
rag_retriever.retrieve("What is MARX file")

retrieving documents for query: What is MARX file
Top K: 5, Score Threshold: 0.0


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.20it/s]


Generated embeddings for 1 texts. Embedding shape: (1, 384)
Retreived 5 documents for query: What is MARX file


[{'id': 'doc_7b10f92c_1661',
  'content': 'data files and reports between Plans and CMS. \nMARx User Guide Link Provides detailed information for using the MARx UI. \nBroadcast Messages Output \nProvides general information about the system’s actions, e.g. \nmonth-end processing started. The list of messages refreshes \nevery time the user returns to the screen. \nCurrent Payment Month \n(CPM) Output The month/year currently in process by the system. \nMARx Version Link The region and release information of the MARx UI display. \nMARx Calendar Link \nProvides general information about what is happening in the \nsystem, e.g. month-end processing started. The list of \nmessages refreshes every time the user returns to the screen.',
  'metadata': {'keywords': 'PCUG',
   'creationdate': '2026-08-31T11:13:53-05:00',
   'subject': 'MAPD PCUG v19.4',
   'content_length': 682,
   'page_label': '650',
   'file_type': 'pdf',
   'total_pages': 790,
   'title': 'MAPD Plan Communication User Guide 

In [17]:
rag_retriever.retrieve("Transaction details for MARX file")

retrieving documents for query: Transaction details for MARX file
Top K: 5, Score Threshold: 0.0


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.49it/s]


Generated embeddings for 1 texts. Embedding shape: (1, 384)
Retreived 5 documents for query: Transaction details for MARX file


[{'id': 'doc_73ec55a4_151',
  'content': 'The following steps are taken to process transactions from a Plan: \n \n• Plans submit transaction files using the selected data exchange method.  \n• MARx processes the submitted transactions, resulting in actions that affect beneficiary \nenrollment, payment, and status. \n• The Plan receives accepted transactions in the Daily Transaction Reply Report (DTRR.  \nThese records contain a Transaction Reply Code (TRC), which describes CMS response.',
  'metadata': {'creationdate': '2026-08-31T11:13:53-05:00',
   'page': 51,
   'source': '..\\data\\pdf\\mapd-plan-communications-user-guide-v19.4-september-2026.pdf',
   'author': 'Elisha Westbrook',
   'keywords': 'PCUG',
   'moddate': '2026-08-31T11:13:53-05:00',
   'total_pages': 790,
   'content_length': 443,
   'page_label': '52',
   'title': 'MAPD Plan Communication User Guide (PCUG) v19.4 September 2026',
   'source_file': 'mapd-plan-communications-user-guide-v19.4-september-2026.pdf',
   'subj

Integration Vectordb Context pipelne with LLM Output


In [61]:
from dotenv import load_dotenv
import requests,os
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_groq import ChatGroq

load_dotenv()

groq_api_key = os.getenv("Grok_API_Key")


llm = ChatGroq(
    groq_api_key=groq_api_key,
    model_name="openai/gpt-oss-20b",
    temperature=0.1,
    max_tokens=1024,
)


def rag_simple(query, retriever, llm, top_k=3):
    # Call your class's retrieve method
    results = retriever.retrieve(query, top_k=top_k)

    # Format context based on what retrieve() returns
    context_blocks = []
    for i, doc in enumerate(results):
        if isinstance(doc, dict):
            content = doc.get("content", doc.get("text", ""))
        elif hasattr(doc, "page_content"):
            content = doc.page_content
        else:
            content = str(doc)

        context_blocks.append(f"Document {i+1}:\n{content}")

    context = "\n\n".join(context_blocks)

    if not context:
        return "No relevant documents found."

    messages = [
        SystemMessage(
            content="You are a helpful assistant. Use the provided context to answer questions accurately and concisely."
        ),
        HumanMessage(content=f"Context:\n{context}\n\nQuestion: {query}"),
    ]

    response = llm.invoke(messages)
    return response.content

In [60]:
import requests, os
resp = requests.get(
    "https://api.groq.com/openai/v1/models",
    headers={"Authorization": f"Bearer {os.getenv('Grok_API_KEY')}"}
)
print([m["id"] for m in resp.json()["data"]])

['whisper-large-v3-turbo', 'openai/gpt-oss-20b', 'meta-llama/llama-prompt-guard-2-22m', 'openai/gpt-oss-120b', 'qwen/qwen3.6-27b', 'openai/gpt-oss-safeguard-20b', 'groq/compound', 'meta-llama/llama-prompt-guard-2-86m', 'allam-2-7b', 'canopylabs/orpheus-v1-english', 'canopylabs/orpheus-arabic-saudi', 'whisper-large-v3', 'groq/compound-mini', 'qwen/qwen3.8-27b']


In [62]:
answer = rag_simple("What is the full name of MARX file?", rag_retriever, llm)
print(answer)

retrieving documents for query: What is the full name of MARX file?
Top K: 3, Score Threshold: 0.0


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.93it/s]


Generated embeddings for 1 texts. Embedding shape: (1, 384)
Retreived 3 documents for query: What is the full name of MARX file?
The full name of the MARX file is **“MARx Data File Monthly.”**


In [66]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("How CMS calculates LEP", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

retrieving documents for query: How CMS calculates LEP
Top K: 3, Score Threshold: 0.1


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.73s/it]


Generated embeddings for 1 texts. Embedding shape: (1, 384)
Retreived 2 documents for query: How CMS calculates LEP
Answer: CMS calculates LEP by multiplying 1 % of the current year’s national base beneficiary Part D premium by the total number of uncovered months (NUNCMO). This calculation is performed annually.
Sources: [{'source': 'mapd-plan-communications-user-guide-v19.4-september-2026.pdf', 'page': 331, 'score': 0.15635812282562256, 'preview': 'of months in which a Medicare-eligible beneficiary did not have creditable drug coverage for a \ncontinuous period of 63 days or more, and report this as the Number of Uncovered Months \n(NUNCMO) to CMS. \n Calculating LEP  \nCMS calculates the LEP by multiplying a percentage, currently 1 percent, of th...'}, {'source': 'mapd-plan-communications-user-guide-v19.4-september-2026.pdf', 'page': 331, 'score': 0.15635812282562256, 'preview': 'of months in which a Medicare-eligible beneficiary did not have creditable drug coverage for a \ncontinu

In [10]:
### Convert text to emeddings
texts=[doc.page_content for doc in chunks]
texts

### Generate the embbeddings



['Version 19.4 \nSeptember 1, 2026',
 'MAPD Plan Communication User Guide Version 19.4 \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \nTHIS PAGE INTENTIONALLY BLANK',
 'MAPD Plan Communication User Guide Version 19.4 \n \nChange Log \n \nSection Change Description \nGlobal • Publish date revised to September 1, 2026 \n1 Introduction: \n• No Change  \n2 Establish Connectivity: \n• No Change \n3 \nEligibility and Enrollment:  \n• Updated Item 10 Position to 47-51 in Layout 3-9.  \n• Updated the Record Length to 600 in Section 3.2.3. \n• Updated Item 10 Position to 121-600 in Section 3.2.4.  \n• Added item 14: Special File – POVER – Incorrect Effective Date in \nSection 3.4.3.  \n4 Low Income Subside (LIS) Status: \n• No Change \n5 Premium: \n• No Change \n6 Payment: \n• Updated the ESRD URL in Section 6.2.2.  \n7 Outbound Files and Miscellaneous: \n• No Change  \n8 MARx UI: \n• No Change \n9 Glossary and Acronyms: \n• Updated the Special Needs Plan (SNP) URL.',
 'MAPD

In [23]:
chunks

[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-08-31T11:13:53-05:00', 'title': 'MAPD Plan Communication User Guide (PCUG) v19.4 September 2026', 'author': 'Elisha Westbrook', 'subject': 'MAPD PCUG v19.4', 'keywords': 'PCUG', 'moddate': '2026-08-31T11:13:53-05:00', 'source': '..\\data\\pdf\\mapd-plan-communications-user-guide-v19.4-september-2026.pdf', 'total_pages': 790, 'page': 0, 'page_label': '1', 'source_file': 'mapd-plan-communications-user-guide-v19.4-september-2026.pdf', 'file_type': 'pdf'}, page_content='Version 19.4 \nSeptember 1, 2026'),
 Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-08-31T11:13:53-05:00', 'title': 'MAPD Plan Communication User Guide (PCUG) v19.4 September 2026', 'author': 'Elisha Westbrook', 'subject': 'MAPD PCUG v19.4', 'keywords': 'PCUG', 'moddate': '2026-08-31T11:13:53-05:00

##Vector Store


In [11]:
class VectorStore:
    def __init__(self,collection_name: str='pdf_documents',persist_directory: str="../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client=None
        self.collection=None
        self.__initialize_store()

    def __initialize_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persist_directory)   
            self.collection=self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description":"PDF document embeddings for RAG"})

            print (f"Vector store initialized with collection: {self.collection_name} at {self.persist_directory}")
            print(f"Existing documents in the collection: {len(self.collection.get()['ids'])}")   
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise 
            
    def add_documents(self, documents: List[str], embeddings:np.ndarray):
        if len(documents)!=len(embeddings):
            raise ValueError("The number of documents and embeddings must be the same.")

        print(f"Adding {len(documents)} documents to the vector store...")
        ids=[]
        metadatas= []
        documents_text=[]
        embeddings_list=[]

        for i,(doc,embeddings) in enumerate(zip(documents,embeddings)):
            doc_id=f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata=dict(doc.metadata)
            metadata['doc_index']=i
            metadata['content_length']=len(doc.page_content)
            metadatas.append(metadata)
            ##Document
            documents_text.append(doc.page_content)
            ##Embeddings
            embeddings_list.append(embeddings.tolist())

        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=documents_text,
                embeddings=embeddings_list
            )
            print(f"Successfully added {len(documents)} documents to the vector store.")  
            print(f"Total documents in the collection after addition: {len(self.collection.get()['ids'])}") 
        except Exception as e:
            print(f"Error adding documents to the vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized with collection: pdf_documents at ../data/vector_store
Existing documents in the collection: 3998


In [12]:

class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name= model_name
        self.model=None
        self._load_model()

    def _load_model(self):
        try:
            print (f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Model is not loaded.")
        try:
            embeddings = self.model.encode(texts, convert_to_numpy=True,show_progress_bar=True)
            print(f"Generated embeddings for {len(texts)} texts. Embedding shape: {embeddings.shape}")
            return embeddings
        except Exception as e:
            print(f"Error generating embeddings: {e}")
            raise  

## intialize the embedding manager
embedding_manager = EmbeddingManager() 
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 843.13it/s]


Model loaded successfully. Embedding dimension: 384


### Embedding and Vector DB

In [5]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
chunks=split_documents(all_pdf_document)
chunks

Split 790 documents into 1999 chunks.

Example chunk:
Content:Version 19.4 
September 1, 2026...
Metadata: {'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-08-31T11:13:53-05:00', 'title': 'MAPD Plan Communication User Guide (PCUG) v19.4 September 2026', 'author': 'Elisha Westbrook', 'subject': 'MAPD PCUG v19.4', 'keywords': 'PCUG', 'moddate': '2026-08-31T11:13:53-05:00', 'source': '..\\data\\pdf\\mapd-plan-communications-user-guide-v19.4-september-2026.pdf', 'total_pages': 790, 'page': 0, 'page_label': '1', 'source_file': 'mapd-plan-communications-user-guide-v19.4-september-2026.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-08-31T11:13:53-05:00', 'title': 'MAPD Plan Communication User Guide (PCUG) v19.4 September 2026', 'author': 'Elisha Westbrook', 'subject': 'MAPD PCUG v19.4', 'keywords': 'PCUG', 'moddate': '2026-08-31T11:13:53-05:00', 'source': '..\\data\\pdf\\mapd-plan-communications-user-guide-v19.4-september-2026.pdf', 'total_pages': 790, 'page': 0, 'page_label': '1', 'source_file': 'mapd-plan-communications-user-guide-v19.4-september-2026.pdf', 'file_type': 'pdf'}, page_content='Version 19.4 \nSeptember 1, 2026'),
 Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-08-31T11:13:53-05:00', 'title': 'MAPD Plan Communication User Guide (PCUG) v19.4 September 2026', 'author': 'Elisha Westbrook', 'subject': 'MAPD PCUG v19.4', 'keywords': 'PCUG', 'moddate': '2026-08-31T11:13:53-05:00

In [3]:
### Teext splitting chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_doc= text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_doc)} chunks.")

    if split_doc:
        print(f"\nExample chunk:") 
        print(f"Content:{split_doc[0].page_content[:200]}...")  # Print the first 500 characters of the first chunk
        print(f"Metadata: {split_doc[0].metadata}")

    return split_doc    


In [13]:
all_pdf_document

[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-08-31T11:13:53-05:00', 'title': 'MAPD Plan Communication User Guide (PCUG) v19.4 September 2026', 'author': 'Elisha Westbrook', 'subject': 'MAPD PCUG v19.4', 'keywords': 'PCUG', 'moddate': '2026-08-31T11:13:53-05:00', 'source': '..\\data\\pdf\\mapd-plan-communications-user-guide-v19.4-september-2026.pdf', 'total_pages': 790, 'page': 0, 'page_label': '1', 'source_file': 'mapd-plan-communications-user-guide-v19.4-september-2026.pdf', 'file_type': 'pdf'}, page_content='Version 19.4 \nSeptember 1, 2026'),
 Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-08-31T11:13:53-05:00', 'title': 'MAPD Plan Communication User Guide (PCUG) v19.4 September 2026', 'author': 'Elisha Westbrook', 'subject': 'MAPD PCUG v19.4', 'keywords': 'PCUG', 'moddate': '2026-08-31T11:13:53-05:00

In [2]:
###Read all the pdfs inside the directory
def process_pdfs_in_directory(directory_path):
    all_documents = []
    pdf_dir=Path(directory_path)
    pdf_files=list(pdf_dir.glob('**/*.pdf'))
    print(f"found {len(pdf_files)} pdf files in the directory {pdf_dir}")
    for pdf_file in pdf_files:
        print(f"Processing file: {pdf_file}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type']='pdf'  # Add the source file path to metadata
            all_documents.extend(documents)
            print(f" Loaded {len(documents)} pages from {pdf_file.name}")
        except Exception as e:
            print(f"Error processing {pdf_file}: {e}")
    print(f"Total documents loaded: {len(all_documents)}")
    return all_documents

#Proccess all PDFs in the directory
all_pdf_document=process_pdfs_in_directory("../data")            


found 1 pdf files in the directory ..\data
Processing file: ..\data\pdf\mapd-plan-communications-user-guide-v19.4-september-2026.pdf
 Loaded 790 pages from mapd-plan-communications-user-guide-v19.4-september-2026.pdf
Total documents loaded: 790


In [1]:
###RAG Pipeline
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader, DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path


C:\Users\DEEPAK MADANA\AppData\Local\Temp\ipykernel_3276\3385670915.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader, DirectoryLoader, TextLoader
e:\AI Agents\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
